## Imports

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.options.display.float_format = '{:,.2f}'.format

## Lista de empresas da B3 e setores

In [3]:
lista_empresas_b3 = pd.read_excel('download_data/ClassifSetorial.xlsx', skiprows = 1).drop('Unnamed: 0', axis = 1)
lista_empresas_b3.columns = ['setor', 'subsetor', 'segmento', 'nome_pregao', 'codigo', 'segmento_negociacao']
lista_empresas_b3 = lista_empresas_b3[lista_empresas_b3['nome_pregao'] != 'NOME DE PREGÃO']
lista_empresas_b3 = lista_empresas_b3[['codigo', 'nome_pregao', 'setor', 'subsetor', 'segmento']]

for c in ['setor', 'subsetor', 'segmento']:
    lista_empresas_b3[c] = lista_empresas_b3[c].ffill()

with pd.option_context('display.max_rows', None):
    display(lista_empresas_b3.head(3))

,codigo,nome_pregao,setor,subsetor,segmento
1,AZTE,AZT ENERGIA,"Petróleo, Gás e Biocombustíveis","Petróleo, Gás e Biocombustíveis","Exploração, Refino e Distribuição"
2,BRAV,BRAVA,"Petróleo, Gás e Biocombustíveis","Petróleo, Gás e Biocombustíveis","Exploração, Refino e Distribuição"
3,CSAN,COSAN,"Petróleo, Gás e Biocombustíveis","Petróleo, Gás e Biocombustíveis","Exploração, Refino e Distribuição"


In [152]:
print(f"Quantidade de linhas da tabela: {lista_empresas_b3.shape[0]}")
print(f"Quantidade de códigos únicos:   {lista_empresas_b3['codigo'].nunique()}")

Quantidade de linhas da tabela: 369
Quantidade de códigos únicos:   369


In [150]:
for c in ['setor', 'subsetor', 'segmento']:
    tmp = lista_empresas_b3[[c]].value_counts().reset_index()
    print(f"{c} | {tmp.shape[0]} categorias")
    display(tmp.head(12))

setor | 11 categorias


,setor,count
0,Consumo Cíclico,89
1,Bens Industriais,55
2,Financeiro,55
3,Utilidade Pública,45
4,Materiais Básicos,32
5,Consumo não Cíclico,28
6,Saúde,22
7,Tecnologia da Informação,15
8,"Petróleo, Gás e Biocombustíveis",13
9,Outros,8


subsetor | 43 categorias


,subsetor,count
0,Energia Elétrica,35
1,Construção Civil,27
2,Intermediários Financeiros,22
3,Transporte,19
4,Comércio Varejista,18
5,"Tecidos, Vestuário e Calçados",15
6,Diversos,14
7,Exploração de Imóveis,13
8,"Petróleo, Gás e Biocombustíveis",13
9,Máquinas e Equipamentos,12


segmento | 83 categorias


,segmento,count
0,Energia Elétrica,35
1,Incorporações,27
2,Bancos,20
3,Programas e Serviços,12
4,Medicamentos e Outros Produtos,11
5,Exploração de Imóveis,11
6,Serviços Diversos,10
7,Serviços Educacionais,10
8,"Exploração, Refino e Distribuição",9
9,Agricultura,9


## Lista de ativos do Ibovespa

In [ ]:
# leitura da base
ativos_ibov = pd.read_csv('download_data/IBOVDia_09-03-26.csv', encoding = 'latin1', sep = ';', skiprows = 1).drop('Part. (%)', axis = 1).reset_index()

# correção do layout
ativos_ibov.columns = ['ticker', 'empresa', 'tipo_acao', 'qtd_teorica', 'pct_part']
ativos_ibov = ativos_ibov[ativos_ibov['empresa'].notna()]

# criar coluna de código para join
ativos_ibov['codigo'] = ativos_ibov['ticker'].apply(lambda x: x[:4])

# separar tipo da ação
ativos_ibov['segm_gov'] = ativos_ibov['tipo_acao'].apply(lambda x: x.split(' ')[-1])
ativos_ibov['tipo_acao_2'] = ativos_ibov.apply(lambda row: row['tipo_acao'].replace(row['segm_gov'], '').strip(), axis = 1)
ativos_ibov['tipo_acao'] = np.where(ativos_ibov['tipo_acao_2'] == '', ativos_ibov['segm_gov'], ativos_ibov['tipo_acao_2'])
ativos_ibov['tipo_acao'] = ativos_ibov['tipo_acao'].str.replace('  ', ' ')
ativos_ibov['segm_gov'] = np.where(ativos_ibov['tipo_acao_2'] == '', None, ativos_ibov['segm_gov'])
ativos_ibov['complemento_tipo'] = ativos_ibov['tipo_acao'].apply(lambda x: x.split(' '))
ativos_ibov['tipo_acao'] = ativos_ibov['complemento_tipo'].apply(lambda x: x[0])
ativos_ibov['complemento_tipo'] = np.where(ativos_ibov['complemento_tipo'].apply(lambda x: len(x)) == 2, ativos_ibov['complemento_tipo'].apply(lambda x: x[-1]), None)
ativos_ibov = ativos_ibov.drop('tipo_acao_2', axis = 1)

# reorganizar colunas
ativos_ibov = ativos_ibov[['ticker', 'codigo', 'empresa', 'tipo_acao', 'complemento_tipo', 'segm_gov', 'qtd_teorica', 'pct_part']]

# transformar colunas float
ativos_ibov['qtd_teorica'] = ativos_ibov['qtd_teorica'].str.replace('.', '').astype(float)
ativos_ibov['pct_part'] = ativos_ibov['pct_part'].str.replace(',', '.').astype(float)

# join com segmento
ativos_ibov = ativos_ibov.merge(lista_empresas_b3.drop('nome_pregao', axis = 1), on = 'codigo', how = 'left')
ativos_ibov = ativos_ibov.drop('codigo', axis = 1)
ativos_ibov.sort_values('pct_part', ascending = False)

,ticker,empresa,tipo_acao,complemento_tipo,segm_gov,qtd_teorica,pct_part,setor,subsetor,segmento
79,VALE3,VALE,ON,NaN,NM,"3,688,870,616.00",11.15,Materiais Básicos,Mineração,Minerais Metálicos
44,ITUB4,ITAUUNIBANCO,PN,EJ,N1,"5,027,870,728.00",8.28,Financeiro,Intermediários Financeiros,Bancos
59,PETR4,PETROBRAS,PN,ATZ,N2,"4,410,960,450.00",7.12,"Petróleo, Gás e Biocombustíveis","Petróleo, Gás e Biocombustíveis","Exploração, Refino e Distribuição"
58,PETR3,PETROBRAS,ON,NaN,N2,"2,943,171,983.00",5.17,"Petróleo, Gás e Biocombustíveis","Petróleo, Gás e Biocombustíveis","Exploração, Refino e Distribuição"
4,AXIA3,AXIA ENERGIA,ON,ATZ,N1,"1,945,846,761.00",4.40,Utilidade Pública,Energia Elétrica,Energia Elétrica
...,...,...,...,...,...,...,...,...,...,...
47,RENT4,LOCALIZA,PN,NaN,NM,"37,577,993.00",0.06,Consumo Cíclico,Diversos,Aluguel de carros
80,VAMO3,VAMOS,ON,NaN,NM,"390,902,974.00",0.06,Consumo Cíclico,Diversos,Aluguel de carros
28,CYRE4,CYRELA REALT,PN,NaN,NM,"48,543,949.00",0.05,Consumo Cíclico,Construção Civil,Incorporações
57,PCAR3,P.ACUCAR-CBD,ON,NaN,NM,"433,412,603.00",0.05,Consumo não Cíclico,Comércio e Distribuição,Alimentos


In [ ]:
for c in ['setor', 'subsetor', 'segmento']:
    tmp = ativos_ibov[[c]].value_counts().reset_index()
    print(f"{c} | {tmp.shape[0]} categorias")
    if tmp.shape[0] <= 10:
        display(tmp)

setor | 10 categorias


,setor,count
0,Consumo Cíclico,16
1,Financeiro,15
2,Utilidade Pública,15
3,Materiais Básicos,10
4,"Petróleo, Gás e Biocombustíveis",9
5,Consumo não Cíclico,7
6,Bens Industriais,5
7,Saúde,5
8,Comunicações,2
9,Tecnologia da Informação,1


subsetor | 28 categorias
segmento | 35 categorias


## Salvar tabela

In [5]:
ativos_ibov.to_excel('refined/ativos_ibov.xlsx', index = False)